importation des libs et chargement des datas

In [65]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px  # Interface haut niveau pour graphiques simples
import plotly.graph_objects as go  # Interface bas niveau pour contrôle précis
import seaborn as sns
from plotly.subplots import make_subplots  # Création de grilles de graphiques

#Affichage complet des colonnes et des lignes (pas de retour a la ligne)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)
pd.set_option("display.expand_frame_repr", False)

#Import du fichier csv customer
url = "./../data/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(url,sep=",")

# Affichage des dimension du jeu de données
print(f"Le jeu de données a {df.shape[0]} lignes et {df.shape[1]} colonnes")

# Affichez les 5 premières lignes
print(df.head())




Le jeu de données a 7043 lignes et 21 colonnes
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService     MultipleLines InternetService OnlineSecurity OnlineBackup DeviceProtection TechSupport StreamingTV StreamingMovies        Contract PaperlessBilling              PaymentMethod  MonthlyCharges TotalCharges Churn
0  7590-VHVEG  Female              0     Yes         No       1           No  No phone service             DSL             No          Yes               No          No          No              No  Month-to-month              Yes           Electronic check           29.85        29.85    No
1  5575-GNVDE    Male              0      No         No      34          Yes                No             DSL            Yes           No              Yes          No          No              No        One year               No               Mailed check           56.95       1889.5    No
2  3668-QPYBK    Male              0      No         No       2          Yes    

Affichez les types de données de chaque colonne

Nombre pour chaque variable

In [66]:
# Affichez les types de données de chaque colonne
print(df.dtypes)
print(" " * 50)
print("-" * 50)
print(" " * 50)
#Nombre de manquants pour chaque colonnes

# remplace les chaînes vides par NaN dans les colonnes de type objet ou string
cols_str = df.select_dtypes(include=["object", "string"]).columns
df[cols_str] = df[cols_str].replace(r"^\s*$", np.nan, regex=True)


if (df.isna().sum() == 0).all():
    print("Il ne manque aucune valeur.")
else:
    print("Il y a des valeurs manquantes.")
    print(f"{df.isna().sum()}")  

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object
                                                  
--------------------------------------------------
                                                  
Il y a des valeurs manquantes.
customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineS

Recherche de doublon

In [67]:
#Recherhce de doublon
print(f"Il y a {df.duplicated().sum()} doublons dans le jeu de données.")
if df.duplicated().sum() > 0:
    print("Voici les lignes en double :")
    print(df[df.duplicated()])

Il y a 0 doublons dans le jeu de données.


Recherche des valeur aberrantes (outlier)
1 Visualisation (boxpplot) sur les valeurs numerique
2 tableau recaptulatif des valeures yes ou no pour les colonnes "Partner", "Dependents", "PhoneService", "PaperlessBilling", "Churn"
        idem pour MultipleLines mais il y a 3 valeurs possible (Yes/No/No phone service)
    si besoin 2 approche
        1 Stat
        2 Machine Learning

In [68]:

# Creaton DataFrame pour ne garder que les valeur numerique pour le boxplot (SeniorCitizen, tenure, MonthlyCharges et un global pour les trois)
df_numeric = df.select_dtypes(include=[np.number])
df_SeniorCitizen = df_numeric["SeniorCitizen"]
df_tenure = df_numeric["tenure"]
df_MonthlyCharges = df_numeric["MonthlyCharges"]
#Creation d'une figure 1 ligne 3 colonnes
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Distribution DSeniorCitizen', 'Distribution Dtenure',' Distribution MonthlyCharges')
)

fig.add_trace(go.Box(y=df_SeniorCitizen, name="SeniorCitizen"), row=1, col=1)
fig.update_xaxes(title_text="SeniorCitizen", row=1, col=1)
fig.update_yaxes(title_text="ancienneté 1/0", row=1, col=1)

fig.add_trace(go.Box(y=df_tenure, name="tenure"), row=1, col=2)
fig.update_xaxes(title_text="tenure", row=1, col=2)
fig.update_yaxes(title_text="Ancienneté client (mois)", row=1, col=2)

fig.add_trace(go.Box(y=df_MonthlyCharges, name="MonthlyCharges"), row=1, col=3)
fig.update_xaxes(title_text="MonthlyCharges", row=1, col=3)
fig.update_yaxes(title_text="Facturation mensuelle", row=1, col=3)


# Affichage du graphique
fig.show()

Recherche de incoherances

In [69]:
inco_tenure = int((df["tenure"] < 0).sum())
inco_monthly = int((df["MonthlyCharges"] < 0).sum())

In [70]:
#Recherche valeur autre que Yes No pour "Partner", "Dependents", "PhoneService", "PaperlessBilling", "Churn"

cols_yes_no = ["Partner", "Dependents", "PhoneService", "PaperlessBilling", "Churn","MultipleLines"]
valeurs_autorisees = ["Yes", "No","No phone service"]

for col in cols_yes_no:
    s = df[col]
    #selection des valeurs non NaN qui ne sont pas dans les valeurs autorisées
    if col == "MultipleLines":
        invalides = s[s.notna() & ~s.astype(str).str.strip().isin(valeurs_autorisees)]
    else:
        #pour les colonnes autre que MultipleLines, les valeurs autorisées sont uniquement "Yes" et "No"
        invalides = s[s.notna() & ~s.astype(str).str.strip().isin(valeurs_autorisees[:2])]
        

    
    if invalides.empty:
        print(f"{col}: OK (NaN={s.isna().sum()})")
    else:
        print(f"{col}: INVALIDES -> {invalides.unique()} (NaN={s.isna().sum()})")



Partner: OK (NaN=0)
Dependents: OK (NaN=0)
PhoneService: OK (NaN=0)
PaperlessBilling: OK (NaN=0)
Churn: OK (NaN=0)
MultipleLines: OK (NaN=0)


Recap du controle qualité

In [74]:
# 1) Valeurs manquantes
missing_counts = df.isna().sum()
missing_total = int(missing_counts.sum())

# 2) Doublons (lignes complètes)
duplicates_count = int(df.duplicated().sum())

# 3) Cohérences métier
inco_tenure = int((df["tenure"] < 0).sum())
inco_monthly = int((df["MonthlyCharges"] < 0).sum())


# 4) Résumé
print("=== Résumé qualité des données ===")
print(f"Valeurs manquantes totales : {missing_total}")
print(f"Doublons (lignes complètes) : {duplicates_count}")
print(f"Incohérences tenure < 0 : {inco_tenure}")
print(f"Incohérences MonthlyCharges < 0 : {inco_monthly}")


print("\nDétail valeurs manquantes (colonnes avec au moins 1 NaN) :")
print(missing_counts[missing_counts > 0].sort_values(ascending=False))

if (
    missing_total == 0
    and duplicates_count == 0
    and inco_tenure == 0
    and inco_monthly == 0
       
    ):
    print("\nConclusion : controles qualite OK.")
else:
    print("\nConclusion : il reste des points a traiter.")

=== Résumé qualité des données ===
Valeurs manquantes totales : 11
Doublons (lignes complètes) : 0
Incohérences tenure < 0 : 0
Incohérences MonthlyCharges < 0 : 0

Détail valeurs manquantes (colonnes avec au moins 1 NaN) :
TotalCharges    11
dtype: int64

Conclusion : il reste des points a traiter.


Exploration de la cible:
    taux Global de churn
    puis par segments 
        Contrat
        InternetService
        tenure
        Payment methode
        gender
        Partner
        Dependents
        PhoneService
        MultipleLines
        InternetService
        OnlineSecurity
        OnlineBackup
        DeviceProtection
        TechSupport
        StreamingTV
        StreamingMovies
        PaperlessBilling
        PaymentMethod
        MonthlyCharges

In [85]:
#Taux global de churn
taux_global_churn = (df["Churn"] == "Yes").mean()
print(f"Taux global de churn : {taux_global_churn:.2%}")

Taux global de churn : 26.54%


In [84]:


# --------------------------
# 0) Base: uniquement résiliés
# --------------------------
resilies = df[df["Churn"] == "Yes"].copy()

# Fonction utilitaire: part des résiliés (%) par modalité
def part_resilies(data, col):
    out = (data[col]
           .value_counts(dropna=False, normalize=True)
           .mul(100)
           .rename_axis(col)
           .reset_index(name="part_pct"))
    return out

# --------------------------
# 1) tenure -> courbe
# --------------------------
tenure_df = part_resilies(resilies, "tenure").sort_values("tenure")
fig_tenure = px.line(
    tenure_df, x="tenure", y="part_pct", markers=True,
    title="Part des résiliés (%) par tenure",
    labels={"part_pct": "Part des résiliés (%)", "tenure": "tenure"}
)
fig_tenure.show()

# --------------------------
# 2) MonthlyCharges -> nuage de points
# --------------------------
mc_df = part_resilies(resilies, "MonthlyCharges").sort_values("MonthlyCharges")
fig_mc = px.scatter(
    mc_df, x="MonthlyCharges", y="part_pct",
    title="Part des résiliés (%) par MonthlyCharges",
    labels={"part_pct": "Part des résiliés (%)", "MonthlyCharges": "MonthlyCharges"}
)
fig_mc.show()

# --------------------------
# 3) Contract + InternetService -> camemberts
# --------------------------
for col in ["Contract", "InternetService"]:
    pie_df = part_resilies(resilies, col)
    fig_pie = px.pie(
        pie_df, names=col, values="part_pct",
        title=f"Répartition des résiliés par {col}"
    )
    fig_pie.show()

# --------------------------
# 4) Segments restants
#    - binaires: 1 seul graphe barres horizontales
#    - autres: 1 bar chart classique par segment
# --------------------------
exclude = {"Churn", "tenure", "MonthlyCharges", "Contract", "InternetService"}

segments = [
    "PaymentMethod", "gender", "Partner", "Dependents", "PhoneService",
    "MultipleLines", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies", "PaperlessBilling"
]
segments = [c for c in segments if c in resilies.columns and c not in exclude]

binary_segments = [c for c in segments if resilies[c].nunique(dropna=False) == 2]
other_segments = [c for c in segments if c not in binary_segments]

# 4a) Un seul graphe horizontal pour les binaires


import plotly.graph_objects as go
import plotly.express as px
import pandas as pd

rows = []
for col in binary_segments:
    tmp = part_resilies(resilies, col).copy()
    tmp["segment"] = col
    tmp = tmp.rename(columns={col: "valeur"})
    tmp["y_label"] = tmp["segment"] + " | " + tmp["valeur"].astype(str)
    rows.append(tmp[["segment", "valeur", "part_pct", "y_label"]])

if rows:
    binary_df = pd.concat(rows, ignore_index=True).sort_values(["segment", "valeur"], kind="stable")

    # Ajout d'une ligne "espace" (unique) entre segments
    out = []
    sep_labels = []
    i = 1
    for seg, g in binary_df.groupby("segment", sort=False):
        out.append(g)
        sep = " " * i   # label unique pour chaque séparateur
        sep_labels.append(sep)
        out.append(pd.DataFrame([{
            "segment": seg, "valeur": "__spacer__", "part_pct": None, "y_label": sep
        }]))
        i += 1

    plot_df = pd.concat(out, ignore_index=True)

    valeurs = [v for v in plot_df["valeur"].unique() if v != "__spacer__"]
    palette = px.colors.qualitative.Set2
    color_map = {v: palette[k % len(palette)] for k, v in enumerate(valeurs)}

    fig_bin = go.Figure(
        go.Bar(
            x=plot_df["part_pct"].fillna(0),
            y=plot_df["y_label"],
            orientation="h",
            marker_color=[
                "rgba(0,0,0,0)" if v == "__spacer__" else color_map[v]
                for v in plot_df["valeur"]
            ],
            customdata=plot_df[["segment", "valeur", "part_pct"]],
            hovertemplate=(
                "Segment: %{customdata[0]}<br>"
                "Valeur: %{customdata[1]}<br>"
                "Part résiliés: %{customdata[2]:.2f}%<extra></extra>"
            ),
        )
    )

    # Lignes séparatrices
    for sep in sep_labels:
        fig_bin.add_shape(
            type="line",
            x0=0, x1=1, xref="paper",
            y0=sep, y1=sep, yref="y",
            line=dict(color="rgba(120,120,120,0.6)", width=1, dash="dot")
        )

    fig_bin.update_layout(
        title="Parts des résiliés (%) - segments binaires",
        xaxis_title="Part des résiliés (%)",
        yaxis_title="Segment / valeur",
        height=max(500, 30 * len(plot_df)),
    )

    fig_bin.show()



# 4b) Bar chart classique pour les autres segments
for col in other_segments:
    tmp = part_resilies(resilies, col).sort_values("part_pct", ascending=False)
    fig = px.bar(
        tmp, x=col, y="part_pct", color=col, text=tmp["part_pct"].round(2),
        title=f"Part des résiliés (%) par {col}",
        labels={"part_pct": "Part des résiliés (%)", col: col}
    )
    fig.update_traces(textposition="outside", showlegend=False)
    fig.show()
